## Tools

Models can request to call tools that perform tasks such as:
- fetching data from a database 
- searching the web
- running code

Tools are pairing of:
- A schema, including the name of the tool, a description, and/or argument definition (often a JSON schema)
- A function or coroutine to execute

In [1]:
import os 
from dotenv import load_dotenv 

load_dotenv()



True

In [3]:
# create model 

from langchain.chat_models import init_chat_model 

model = init_chat_model(model = "groq:qwen/qwen3-32b")
response = model.invoke("what is AI?")
response

AIMessage(content='<think>\nOkay, so the user is asking, "what is AI?" Let me start by recalling what I know about AI. AI stands for Artificial Intelligence, right? But I need to make sure I explain it in a way that\'s clear and not too technical.\n\nFirst, maybe I should define it in simple terms. Artificial Intelligence is the simulation of human intelligence processes by machines, especially computer systems. But how do I break that down? Maybe mention things like learning, reasoning, problem-solving, perception, and language understanding. Oh, and there are different types of AI, like narrow AI versus general AI. Narrow AI is what we see today, like voice assistants or recommendation systems. General AI would be more like human-level intelligence across all tasks, which we don\'t have yet.\n\nI should also mention the key components or techniques used in AI. Machine Learning is a big part of it, where systems learn from data. There\'s Deep Learning, which uses neural networks with 

In [24]:
# create ad bind tools

from langchain.tools import tool 

@tool
def get_weather(location:str) -> str:
    """Get the weather at a location """
    return f"It is sunny in {location}"

@tool 
def get_time(location:str) -> str:
    """Get the current time at a location"""
    return f"The current time in {location} is 10:00 AM"

tools = [get_weather, get_time]



# bind tool with llm (we can also use create_agent())
model_with_tools = model.bind_tools([get_weather])
model_with_tools

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x00000201FBC835C0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000201FAC1D790>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather at a location', 'parameters': {'properties': {'location': {'type': 'string'}}, 'required': ['location'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [7]:
response = model_with_tools.invoke("what's the weather in coimbatore")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call["name"]}")
    print(f"Args: {tool_call["args"]}")

content='' additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Coimbatore. I need to use the get_weather function. The function requires the location parameter. Coimbatore is a city in Tamil Nadu, India. So I should pass "Coimbatore" as the location. Let me check if there are any other parameters needed, but the tool only asks for location. Alright, I\'ll format the tool call with the name as get_weather and arguments with location set to Coimbatore.\n', 'tool_calls': [{'id': 'twqxc0618', 'function': {'arguments': '{"location":"Coimbatore"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 129, 'prompt_tokens': 155, 'total_tokens': 284, 'completion_time': 0.208740547, 'completion_tokens_details': {'reasoning_tokens': 102}, 'prompt_time': 0.00873555, 'prompt_tokens_details': None, 'queue_time': 0.160675228, 'total_time': 0.217476097}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d3

In [25]:
# Create a mapping from tool name -> tool object


tool_registry = {tool.name: tool for tool in tools}
tool_registry

{'get_weather': StructuredTool(name='get_weather', description='Get the weather at a location', args_schema=<class 'langchain_core.utils.pydantic.get_weather'>, func=<function get_weather at 0x00000201FC31D260>),
 'get_time': StructuredTool(name='get_time', description='Get the current time at a location', args_schema=<class 'langchain_core.utils.pydantic.get_time'>, func=<function get_time at 0x00000201FC31D120>)}

## Tool execution loops

In [27]:
# step 1: model generates tool calls 
messages = [{"role": "user", "content": "What's the weather in coimbatore?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

print(messages)

# step 2: execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_name = tool_call["name"]
    selected_tool = tool_registry[tool_name]
    tool_result = selected_tool.invoke(tool_call)
    messages.append(tool_result)

# step 3: pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response)

[{'role': 'user', 'content': "What's the weather in coimbatore?"}, AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Coimbatore. I need to use the get_weather function. The function requires a location parameter. Coimbatore is a city in Tamil Nadu, India. So I should pass "Coimbatore" as the location. Let me check if there are any other parameters needed, but the tool only asks for location. Alright, I\'ll format the tool call with the name as get_weather and arguments with location set to Coimbatore. Make sure the JSON is correctly structured.\n', 'tool_calls': [{'id': 'qmyg4jc3j', 'function': {'arguments': '{"location":"Coimbatore"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 137, 'prompt_tokens': 156, 'total_tokens': 293, 'completion_time': 0.2327612, 'completion_tokens_details': {'reasoning_tokens': 110}, 'prompt_time': 0.006335449, 'prompt_tokens_details': None, 'q